In [76]:
import os
from dotenv import load_dotenv
load_dotenv()

token = os.getenv("RPT_TOKEN")
url = os.getenv("RPT_URL")

In [77]:
import requests
import json

# Configuration
API_TOKEN = token 
API_URL = "https://rpt.cloud.sap/api/predict" # or just url

def predict_with_rpt1(rows, index_column=None, api_token=API_TOKEN, api_url = "https://rpt.cloud.sap/api/predict"):
    """
    Make predictions using SAP-RPT-1 Model API
    
    Args:
        rows (list): List of dictionaries representing data rows
                    Context rows should have complete data
                    Query rows should have '[PREDICT]' for values to predict
        index_column (str, optional): Column name to use as row identifier
        api_token (str): Your API authentication token
    
    Returns:
        dict: Prediction results from the API
    
    Constraints:
        - Minimum 1 query row (rows with [PREDICT])
        - Maximum 25 query rows
        - Minimum 2 context rows (rows without [PREDICT])
        - Maximum 2,048 context rows
        - Maximum 4 target columns (columns containing [PREDICT])
        - Maximum 50 columns total per row
        - Maximum 2,073 rows total
        - Maximum 1,000 characters per cell
        - Maximum 100 characters per column name
    """
    
    # Prepare request payload
    payload = {
        "rows": rows
    }
    
    if index_column:
        payload["index_column"] = index_column
    
    # Prepare headers
    headers = {
        "Authorization": f"Bearer {api_token}",
        "Content-Type": "application/json"
    }
    
    try:
        # Make API request
        response = requests.post(api_url, headers=headers, json=payload)
        
        # Check for rate limiting
        if response.status_code == 429:
            retry_after = response.headers.get('Retry-After', 'unknown')
            print(f"Rate limit exceeded. Retry after {retry_after} seconds.")
            return None
        
        # Check for service unavailability
        if response.status_code == 503:
            retry_after = response.headers.get('Retry-After', 'unknown')
            print(f"Service unavailable. Retry after {retry_after} seconds.")
            return None
        
        # Raise error for bad status codes
        response.raise_for_status()
        
        # Return parsed response
        result = response.json()
        return result
        
    except requests.exceptions.RequestException as e:
        print(f"API request failed: {e}")
        if hasattr(e.response, 'text'):
            print(f"Response: {e.response.text}")
        return None

In [78]:
# Example 1: Single column prediction (Churn prediction)
example_churn = [
    {
        "customer_id": "C001",
        "age": 35,
        "subscription_months": 12,
        "churn": "No"
    },
    {
        "customer_id": "C042",
        "age": 20,
        "subscription_months": 1,
        "churn": "Yes"
    },
    {
        "customer_id": "C002",
        "age": 28,
        "subscription_months": 3,
        "churn": "[PREDICT]"
    },
    {
        "customer_id": "C055",
        "age": 55,
        "subscription_months": 1,
        "churn": "[PREDICT]"
    }
]
result = predict_with_rpt1(example_churn, index_column="customer_id", api_token=API_TOKEN)
if result:
    print(json.dumps(result, indent=2))

column_of_interest = 'churn'
result['aiApiResponsePayload']['predictions'][0][column_of_interest][0]['prediction']

{
  "prediction": {
    "id": "ad1eeb85-4e1f-48f9-9004-3779fde85b88",
    "metadata": {
      "num_columns": 3,
      "num_predict_rows": 2,
      "num_predict_tokens": 2,
      "num_rows": 2
    },
    "predictions": [
      {
        "churn": [
          {
            "confidence": null,
            "prediction": "Yes"
          }
        ],
        "customer_id": "C002"
      },
      {
        "churn": [
          {
            "confidence": null,
            "prediction": "Yes"
          }
        ],
        "customer_id": "C055"
      }
    ]
  },
  "delay": 236.50944897532463,
  "aiApiRequestPayload": {
    "prediction_config": {
      "target_columns": [
        {
          "name": "churn",
          "placeholder_value": "[PREDICT]",
          "task_type": "classification"
        }
      ]
    },
    "rows": [
      {
        "customer_id": "C001",
        "age": 35,
        "subscription_months": 12,
        "churn": "No"
      },
      {
        "customer_id": "C042",
      

'Yes'

In [79]:
result['aiApiResponsePayload']['predictions']

[{'churn': [{'confidence': None, 'prediction': 'Yes'}], 'customer_id': 'C002'},
 {'churn': [{'confidence': None, 'prediction': 'Yes'}], 'customer_id': 'C055'}]

In [80]:
# Example 2: Multiple column prediction (Quantity and Revenue)
example_multi = [ # df.to_dict(orient="records")
    {
        "product": "Widget A",
        "price": 29.99,
        "quantity": "[PREDICT]",
        "revenue": "[PREDICT]"
    },
    {
        "product": "Widget B",
        "price": 25.95,
        "quantity": 2,
        "revenue": 10025.78
    },
    {
        "product": "Widget C",
        "price": 39.99,
        "quantity": 150,
        "revenue": 5998.50
    }
]

import pandas as pd
pd.DataFrame(example_multi)

,product,price,quantity,revenue
0,Widget A,29.99,[PREDICT],[PREDICT]
1,Widget B,25.95,2,10025.78
2,Widget C,39.99,150,5998.5


In [81]:
# API Call

result = predict_with_rpt1(example_multi, api_token=API_TOKEN)
if result:
    print(json.dumps(result, indent=2))

{
  "prediction": {
    "id": "a145b1e4-004d-4b32-bb0b-c52d8a24abc6",
    "metadata": {
      "num_columns": 4,
      "num_predict_rows": 1,
      "num_predict_tokens": 2,
      "num_rows": 2
    },
    "predictions": [
      {
        "quantity": [
          {
            "confidence": null,
            "prediction": 79.86937801837922
          }
        ],
        "revenue": [
          {
            "confidence": null,
            "prediction": 8603.481846692017
          }
        ]
      }
    ]
  },
  "delay": 291.61927899718285,
  "aiApiRequestPayload": {
    "prediction_config": {
      "target_columns": [
        {
          "name": "quantity",
          "placeholder_value": "[PREDICT]",
          "task_type": "regression"
        },
        {
          "name": "revenue",
          "placeholder_value": "[PREDICT]",
          "task_type": "regression"
        }
      ]
    },
    "rows": [
      {
        "product": "Widget A",
        "price": 29.99,
        "quantity": "[PRED

In [82]:
column_of_interest = 'quantity'
result['aiApiResponsePayload']['predictions'][0][column_of_interest][0]['prediction']

79.86937801837922

In [83]:
column_of_interest = 'revenue'
result['aiApiResponsePayload']['predictions'][0][column_of_interest][0]['prediction']

8603.481846692017